# 03 — Train IsolationForest

Unsupervised anomaly detection on 6 sensor features. The model learns what 'normal'
looks like; deviations get flagged for operator review.


In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import warnings; warnings.filterwarnings('ignore')

df = pd.read_csv('../data/raw/hot_corridor.csv')
FEATURES = ['inlet_temp_c','outlet_temp_c','power_kw','airflow_cfm','humidity_pct','cpu_util_pct']
avail = [c for c in FEATURES if c in df.columns]
X = df[avail].fillna(df[avail].median())
print(f'Training: {len(X)} samples, features={avail}')


In [ ]:
# Scale + train
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
joblib.dump(scaler, '../artifacts/scaler_v1.joblib')

model = IsolationForest(n_estimators=200, contamination=0.05, max_samples=0.8, random_state=42, n_jobs=-1)
model.fit(X_scaled)
joblib.dump(model, '../artifacts/isolation_forest_v1.joblib')
scores = model.decision_function(X_scaled)
print(f'Mean score: {scores.mean():.4f}  |  Std: {scores.std():.4f}')


## Interpreting Scores
- More negative = more anomalous
- Threshold ~-0.15 separates anomalies from normal
- Scores feed directly into RiskScorer ensemble (50% weight)
